# 07 — Speed: export + benchmark

Exports the trained model to **ONNX** and **TensorRT FP16**, then benchmarks **imgsz 640 / 512 / 416** across the available backends and prints a comparison table.

**FPS is measured with a real timer around the FULL loop** — frame read + inference + ByteTrack + draw — not the model call alone. The target is **≥ 30 FPS end-to-end**; the verdict cell flags any config that misses it honestly (no model-only latency quoting).

Two classes cost nothing measurable over one at this scale — the head is a handful of extra channels — so these numbers should land close to `croprow/`'s. If they do not, suspect the source clip or the machine, not the class count.

> Ships **unrun** — needs a trained checkpoint and a source video. TensorRT export/benchmark requires an NVIDIA GPU; it is skipped cleanly on CPU-only machines.

In [ ]:
import os, sys
from pathlib import Path

# Locate repo root (holds croprow_disease/utils.py) so the package imports
# regardless of the cwd the notebook is launched from.
REPO_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "croprow_disease" / "utils.py").is_file()
)
sys.path.insert(0, str(REPO_ROOT))
from croprow_disease import utils as U
from croprow_disease.health import HealthParams

CW = REPO_ROOT / "croprow_disease"
DATA_DIR = CW / "data"
MODELS_DIR = CW / "models"
RUNS_DIR = CW / "runs"
RESULTS_MD = CW / "RESULTS.md"

import cv2

# ===================== CONFIG (edit here only) =====================
WEIGHTS      = str(MODELS_DIR / "best.pt")
IMGSZ_LIST   = [640, 512, 416]
SOURCE_VIDEO = str(RUNS_DIR / "seq_test_0003.mp4")   # make one in 06 Part 1
DEVICE       = 0
HALF         = True          # FP16 (GPU)
CONF, IOU    = 0.25, 0.50
TRACKER      = "bytetrack.yaml"
N_WARMUP     = 10            # warm-up frames excluded from timing
N_FRAMES     = 200           # timed frames (loops video if shorter)
TARGET_FPS   = 30
# ===================================================================
RUNS_DIR.mkdir(parents=True, exist_ok=True)
print("weights:", WEIGHTS)
print("source :", SOURCE_VIDEO)

## Environment check

In [ ]:
# This notebook needs the training/inference stack (torch + ultralytics),
# NOT installed in the light 01/02 env. See croprow_disease/requirements-train.txt.
try:
    import torch
    from ultralytics import YOLO
    import ultralytics
    print("torch      :", torch.__version__)
    print("ultralytics:", ultralytics.__version__)
    print("CUDA avail :", torch.cuda.is_available(),
          "|", (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only"))
except ModuleNotFoundError as e:
    raise ModuleNotFoundError(
        f"Missing training dependency: {e.name}. Install "
        "croprow_disease/requirements-train.txt into the croprow Python 3.11 venv "
        "before running this notebook."
    ) from e

## Export to ONNX and TensorRT (per imgsz)

One ONNX + one TensorRT engine per image size. TensorRT is attempted only when CUDA is available.

In [ ]:
import torch

if not Path(WEIGHTS).is_file():
    raise FileNotFoundError(f"Weights not found: {WEIGHTS}. Train first (03/04).")
if not Path(SOURCE_VIDEO).is_file():
    raise FileNotFoundError(
        f"Source video not found: {SOURCE_VIDEO}. Make one with 06 Part 1, or "
        "point SOURCE_VIDEO at any mp4.")

cuda = torch.cuda.is_available()
exports = {}   # imgsz -> {"pytorch": .pt, "onnx": .onnx, "engine": .engine|None}
for sz in IMGSZ_LIST:
    entry = {"pytorch": WEIGHTS}
    m = YOLO(WEIGHTS)
    entry["onnx"] = m.export(format="onnx", imgsz=sz, half=HALF and cuda,
                             dynamic=False, device=DEVICE if cuda else "cpu")
    if cuda:
        try:
            entry["engine"] = YOLO(WEIGHTS).export(format="engine", imgsz=sz,
                                                   half=HALF, device=DEVICE)
        except Exception as e:
            print(f"[imgsz {sz}] TensorRT export failed: {e}")
            entry["engine"] = None
    else:
        entry["engine"] = None
        print(f"[imgsz {sz}] CUDA unavailable -> skipping TensorRT engine")
    exports[sz] = entry
    print(f"imgsz {sz}: onnx={entry['onnx']} engine={entry['engine']}")

## Benchmark the FULL end-to-end loop

In [ ]:
import time

def bench(weights_path, imgsz):
    """Mean end-to-end FPS: read + track + draw over N_FRAMES timed frames."""
    model = YOLO(str(weights_path))
    cap = cv2.VideoCapture(SOURCE_VIDEO)
    if not cap.isOpened():
        raise FileNotFoundError(f"cannot open source video: {SOURCE_VIDEO}")

    done, warm, t_all = 0, 0, 0.0
    while done < N_FRAMES:
        ok, frame = cap.read()
        if not ok:  # loop the clip if it is shorter than N_FRAMES
            cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
            ok, frame = cap.read()
            if not ok:
                break
        t0 = time.perf_counter()
        res = model.track(frame, persist=True, tracker=TRACKER, imgsz=imgsz,
                          conf=CONF, iou=IOU, device=DEVICE if cuda else "cpu",
                          verbose=False)[0]
        b = res.boxes
        if b is not None and b.xyxy is not None and len(b):
            xyxy = b.xyxy.cpu().numpy().astype(int)
            clss = b.cls.cpu().numpy().astype(int)
            for (x1, y1, x2, y2), c in zip(xyxy, clss):
                cv2.rectangle(frame, (x1, y1), (x2, y2),
                              U.CLASS_COLORS.get(int(c), (200, 200, 200)), 2)
        dt = time.perf_counter() - t0
        if warm < N_WARMUP:
            warm += 1
            continue
        t_all += dt
        done += 1
    cap.release()
    return (done / t_all) if t_all > 0 else 0.0

rows = []
for sz in IMGSZ_LIST:
    for backend in ("pytorch", "onnx", "engine"):
        wpath = exports[sz][backend]
        if not wpath:
            continue
        fps = bench(wpath, sz)
        rows.append((backend, sz, round(fps, 1),
                     "PASS" if fps >= TARGET_FPS else "MISS"))
        print(f"{backend:8s} imgsz {sz}: {fps:6.1f} FPS  "
              f"{'PASS' if fps >= TARGET_FPS else 'MISS'}")

## Comparison table + honest verdict

In [ ]:
import pandas as pd

df = pd.DataFrame(rows, columns=["backend", "imgsz", "end_to_end_fps", "vs_30fps"])
df = df.sort_values("end_to_end_fps", ascending=False).reset_index(drop=True)
print(df.to_string(index=False))

ok = df[df["end_to_end_fps"] >= TARGET_FPS]
print()
if len(ok):
    best = ok.iloc[0]
    print(f"MEETS {TARGET_FPS} FPS target: {len(ok)} config(s). "
          f"Fastest: {best['backend']} @ imgsz {best['imgsz']} "
          f"= {best['end_to_end_fps']} FPS end-to-end.")
else:
    print(f"NO config meets the {TARGET_FPS} FPS end-to-end target on this "
          f"machine. Reporting measured numbers as-is (full loop, not "
          f"model-only). Consider a smaller imgsz, a GPU, or TensorRT FP16.")

df